# Notebook 07: Deep Learning Data Pipeline & Verification Checks
**Project 26 — Card Fraud Triage (GMM vs. Autoencoder) — Team T033**  
**Author:** Krishna Pramod Dubale (Task P2_01)  
**Phase:** Phase 2 (Deep Learning & Hybrid Anomaly Detection)  

---

### Objectives
1. **PyTorch Tensor Construction:** Convert Phase 1 chronological partitions into `torch.FloatTensor` objects.
2. **Strict Legit-Only Invariant Enforcement:** Mathematically assert that zero fraud transactions enter training tensors or DataLoaders.
3. **Anti-Leakage Early Stopping:** Validate that early stopping monitors reconstruction loss strictly on legitimate validation records (`X_val_legit`).
4. **Feature Variance Dominance Analysis:** Demonstrate why raw coordinate variances must be standardized to prevent high-variance features (e.g., `Amount`) from monopolizing the MSE reconstruction error.
5. **DataLoader Performance Benchmarking:** Benchmark batch streaming throughput, epoch iteration latency, and memory utilization for mini-batch training (`batch_size=512`).
6. **Diagnostic Visualizations:** Generate and audit all Phase 2 data pipeline figures.

In [1]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# Add repo root to path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import CFG, get_repo_root
from src.data import Splits, get_features
from src.eda_dl import (
    create_dl_tensors,
    create_dl_dataloaders,
    audit_dl_leakage_and_invariants,
    analyze_feature_variances_and_dominance,
    benchmark_dl_dataloaders,
    plot_dl_data_diagnostics,
)
from src.utils import set_seed

set_seed(int(CFG.get("seed", 42)))
print(f"Project Root: {ROOT}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()} | MPS Available: {torch.backends.mps.is_available()}")

Project Root: /Users/krishnapramoddubale/Desktop/DL project/T033-CardFraudDetection-AML-DL
PyTorch Version: 2.9.1
CUDA Available: False | MPS Available: True


In [2]:
# 1. Load canonical Phase 1 splits (feature_set='base')
splits = get_features(feature_set="base")

print("── Dataset Partition Summary ──")
print(f"Train Legit: {splits.X_train_legit.shape} (pure legitimate)")
print(f"Train All:   {splits.X_train_all.shape} (for diagnostics only)")
print(f"Val:         {splits.X_val.shape} (frauds: {(splits.y_val == 1).sum()})")
print(f"Test:        {splits.X_test.shape} (frauds: {(splits.y_test == 1).sum()})")
print(f"Total features: {len(splits.feature_names)}")

── Dataset Partition Summary ──
Train Legit: (170524, 31) (pure legitimate)
Train All:   (170884, 31) (for diagnostics only)
Val:         (56961, 31) (frauds: 57)
Test:        (56962, 31) (frauds: 75)
Total features: 31


In [3]:
# 2. Convert to PyTorch FloatTensors
tensors = create_dl_tensors(splits)

# Invariant assertions
assert (splits.y_train_all == 0).sum() == len(tensors["X_train_legit"]), "Leakage: Training rows mismatch!"
assert torch.isfinite(tensors["X_train_legit"]).all(), "Training tensor contains NaN/Inf!"
assert torch.isfinite(tensors["X_val_legit"]).all(), "Val legit tensor contains NaN/Inf!"

print("All Tensor Invariant Assertions PASSED successfully.")
print(f"X_train_legit Tensor: shape={tensors['X_train_legit'].shape}, dtype={tensors['X_train_legit'].dtype}")
print(f"X_val_legit Tensor:   shape={tensors['X_val_legit'].shape}, dtype={tensors['X_val_legit'].dtype}")

All Tensor Invariant Assertions PASSED successfully.
X_train_legit Tensor: shape=torch.Size([170524, 31]), dtype=torch.float32
X_val_legit Tensor:   shape=torch.Size([56904, 31]), dtype=torch.float32


In [4]:
# 3. Build PyTorch DataLoaders (batch_size=512)
loaders = create_dl_dataloaders(tensors, batch_size=512, seed=int(CFG.get("seed", 42)))

# Inspect first mini-batch from train_loader
batch_x, batch_target = next(iter(loaders["train_loader"]))

print(f"Train DataLoader Batches: {len(loaders['train_loader'])}")
print(f"First Mini-Batch Input:  shape={batch_x.shape}, dtype={batch_x.dtype}")
print(f"First Mini-Batch Target: shape={batch_target.shape}, dtype={batch_target.dtype}")
assert torch.equal(batch_x, batch_target), "Autoencoder target must be identical to clean input!"
print("Mini-batch unsupervised reconstruction invariant verified: target == input.")

Train DataLoader Batches: 334
First Mini-Batch Input:  shape=torch.Size([512, 31]), dtype=torch.float32
First Mini-Batch Target: shape=torch.Size([512, 31]), dtype=torch.float32
Mini-batch unsupervised reconstruction invariant verified: target == input.


In [5]:
# 4. Execute Phase 2 Data Leakage Audit
audit_df = audit_dl_leakage_and_invariants(splits)
pd.set_option("display.max_colwidth", None)
display(audit_df[["Check Item", "Observed Value", "Status", "Mathematical Justification"]])

Check Item,Observed Value,Status,Mathematical Justification
Training Set Legit-Only Isolation,170524 legit records (0 fraud),PASS,Unsupervised one-class formulation models normal manifold without anomaly distortion.
StandardScaler Training Source,Diff on V1 mean: 0.00e+00,PASS,Pre-training normalizer cannot absorb validation or test statistics.
Chronological Partition Invariance,Train max=120396s <= Val min=120396s <= Test min=145248s,PASS,Prevents future transaction memorization; mimics production deployment.
Early-Stopping Loss Hygiene,56904 val-legit rows isolated (57 frauds excluded),PASS,Fraud labels are not used to select training stopping epoch.
No Synthetic Oversampling (SMOTE),0 synthetic records added,PASS,"SMOTE interpolates non-existent frauds, destroying rare boundary geometry."


In [6]:
# 5. Variance Dominance & Coordinate Scale Analysis
var_df, stat_df = analyze_feature_variances_and_dominance(splits)

print("Top 10 Features by Share of Total MSE Loss (Unscaled vs. Scaled):")
display(var_df[["Feature", "Unscaled Var", "Unscaled MSE Weight (%)", "Scaled Train Var", "Scaled Train MSE Weight (%)", "Val Variance Ratio (Val/Train)"]].head(10))

Top 10 Features by Share of Total MSE Loss (Unscaled vs. Scaled):


Feature,Unscaled Var,Unscaled MSE Weight (%),Scaled Train Var,Scaled Train MSE Weight (%),Val Variance Ratio (Val/Train)
V1,3.243403,10.227503,0.999988,3.225951,1.451761
V2,2.516602,7.935662,0.999936,3.225781,1.261732
V3,1.630513,5.141536,0.999968,3.225886,1.219478
V4,1.819131,5.736309,0.999961,3.225863,1.239893
V5,1.690292,5.330040,0.999953,3.225838,1.191893
V6,1.667641,5.258614,0.999954,3.225841,1.165356
V7,1.232254,3.885697,0.999944,3.225807,1.313119
V8,1.379303,4.349390,0.999882,3.225607,0.958447
V9,1.297282,4.090752,0.999959,3.225856,0.818380
V10,1.055448,3.328171,0.999925,3.225748,1.106605


In [7]:
# 6. Benchmark DataLoader Throughput and Memory Utilization
bench_df = benchmark_dl_dataloaders(loaders, batch_size=512)
display(bench_df)

Dataset Split,Samples,Features,Batch Size,Total Batches,Memory (MB),Streaming Latency (ms/batch),Throughput (samples/sec),Est. Full Epoch Stream Time (s)
Train (Legitimate Only),170524,31,512,334,20.17,0.89,575385.1,0.296
Validation (Legit-Only Early Stop),56904,31,512,112,6.73,0.89,575385.1,0.099
Test (Evaluation),56962,31,512,112,6.74,0.89,575385.1,0.099


In [8]:
# 7. Generate and display diagnostic figures
plot_dl_data_diagnostics(splits, var_df, bench_df)

fig_paths = [
    ROOT / "results/figures/dl_data_feature_variances.png",
    ROOT / "results/figures/dl_data_batch_composition.png",
    ROOT / "results/figures/dl_data_reconstruction_impact.png",
    ROOT / "results/figures/dl_data_norm_distributions.png",
]

for p in fig_paths:
    print(f"Verified Figure Exists: {p.name} ({p.stat().st_size:,} bytes)")

Verified Figure Exists: dl_data_feature_variances.png (164,244 bytes)
Verified Figure Exists: dl_data_batch_composition.png (183,120 bytes)
Verified Figure Exists: dl_data_reconstruction_impact.png (188,416 bytes)
Verified Figure Exists: dl_data_norm_distributions.png (191,728 bytes)


## 8. Summary of Findings & Viva Defense Points

### 1. Why Legit-Only Training?
In retail card fraud triage, genuine frauds represent only ~0.17% of all operations. Ground-truth fraud labels are not available at transaction time, emerging only after a 30–60 day cardholder verification lag. Training exclusively on legitimate transactions allows the Undercomplete Autoencoder to learn the nominal distribution and data manifold geometry without overfitting to historical adversarial patterns.

### 2. Why Validation-Legit Early Stopping?
If early stopping were governed by validation PR-AUC or mixed validation loss, the model checkpoint selection would actively exploit fraud labels, introducing covert supervisory feedback. By monitoring **validation-legitimate reconstruction loss**, early stopping identifies the epoch where representation learning on nominal data reaches optimal convergence, completely preserving the validation fraud labels for threshold selection.

### 3. Why Feature Normalization is Critical for Autoencoder MSE?
The Mean Squared Error objective is coordinate-isotropic:
$$\mathcal{L}(\mathbf{x}, \hat{\mathbf{x}}) = rac{1}{d} \sum_{j=1}^d (x_j - \hat{x}_j)^2$$
Without standard scaling, high-variance coordinates (e.g. raw transaction amounts with variance $\sigma^2 pprox 62,500$) dominate the gradient magnitude by orders of magnitude, effectively blinding the autoencoder to deviations across the remaining 28 latent features. Standard scaling ensures that all coordinates contribute equally ($1/d pprox 3.23\%$) in nominal expected loss.

### 4. Why Oversampling (SMOTE) is Flawed in One-Class Triage
SMOTE synthesizes minority cases via convex combinations $	ilde{\mathbf{x}} = \mathbf{x}_i + \lambda (\mathbf{x}_{zi} - \mathbf{x}_i)$. For unsupervised fraud triage, this is doubly invalid:
- Synthesizing non-existent frauds violates the one-class premise where the model only models normality.
- Linear interpolation between complex fraudulent events creates artificial points in non-fraudulent regions, degrading boundary fidelity.